# 14. Graph ML — Fraud Network Detection

## Business Case

Traditional fraud detection often evaluates a transaction independently:

```text
Transaction
    ↓
Amount
Time
Channel
Customer features
    ↓
Fraud / Normal
```

But fraud can be **networked**.

A suspicious account may not look unusual by itself, but its relationships with other accounts can reveal:

- fraud rings,
- mule accounts,
- circular transfers,
- coordinated activity,
- shared beneficiaries,
- unusual money-flow structures.

Graph Machine Learning represents the banking ecosystem as a **network**.

> This notebook uses synthetic transaction data for educational purposes only.

## 1. What is a Graph?

A graph consists of:

```text
Nodes + Edges
```

### Nodes

Bank accounts:

```text
A001
A002
A003
```

### Edges

Transactions:

```text
A001 → A002
A002 → A003
A003 → A001
```

Additional information can be attached to nodes and edges.

```text
Node features:
- account age
- transaction count
- balance

Edge features:
- amount
- timestamp
- channel
```

## 2. Why Graphs Help Fraud Detection

Consider:

```text
A → B
A → C
B → D
C → D
D → A
```

Individually, each transaction may appear normal.

Together, the structure may indicate coordinated behavior.

Graph analytics can therefore add a relational dimension:

```text
Traditional ML
"What does this transaction look like?"

Graph ML
"What does this transaction/account look like
relative to the surrounding network?"


## 3. Banking Graph Fraud Use Cases

### Account Network

```text
Account → Account
```

### Merchant Network

```text
Customer → Merchant
```

### Payment Network

```text
Sender → Receiver
```

### Device Network

```text
Customer → Device
```

### Identity Network

```text
Customer → Phone / Email / Address
```

A heterogeneous graph can combine several entity types.

## 4. Project Objective

The objective is restated in one place: detect fraud and mule-account patterns from the structure of money flows, not from single-transaction values alone.


We will build an educational end-to-end pipeline:

```text
Transaction Data
      ↓
Graph Construction
      ↓
Network Exploration
      ↓
Node / Edge Features
      ↓
Network Metrics
      ↓
Fraud Risk Score
      ↓
Community Detection
      ↓
Suspicious Network Identification
      ↓
Graph ML Concept
      ↓
Business Action
```

The notebook demonstrates graph concepts with practical Python tools and a lightweight anomaly/risk model.

## 5. Import Libraries

This step imports the libraries used throughout the notebook — pandas and NumPy for data handling, NetworkX for graph construction and network metrics, and scikit-learn for the downstream classifier.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score
)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns",100)

tx=pd.read_csv("bank_fraud_network_transactions_sample.csv")

print("Transactions:",len(tx))
print("Accounts:",pd.unique(
    tx[["Source_Account","Target_Account"]].values.ravel()
).size)

display(tx.head())

## 6. Dataset Dictionary

Accounts, transactions and known fraud labels are documented, fixing the columns from which the graph's nodes and edges will be built.


In [ ]:
dictionary=pd.DataFrame({
    "Column":[
        "Transaction_ID","Source_Account","Target_Account",
        "Amount","Hour","Channel","Fraud_Label"
    ],
    "Meaning":[
        "Transaction identifier",
        "Account sending funds",
        "Account receiving funds",
        "Transaction amount",
        "Transaction hour",
        "Transaction channel",
        "Synthetic fraud indicator"
    ]
})
display(dictionary)

## 7. Basic Data Quality

Missing values, duplicate transaction IDs and impossible amounts are checked before the graph is built, because every bad row becomes a wrong edge that pollutes the whole network.


In [ ]:
print("Missing values:")
display(tx.isna().sum().to_frame("Missing"))

print("Duplicate transactions:",tx["Transaction_ID"].duplicated().sum())

display(tx["Channel"].value_counts().to_frame("Transactions"))

print("Fraud rate:",round(tx["Fraud_Label"].mean(),4))

## 8. Build the Transaction Graph

For this project:

```text
Node = Account
Edge = Transaction
```

Because multiple transactions can occur between the same accounts, we use a **directed graph** and retain transaction-level attributes separately.

In [ ]:
G=nx.DiGraph()

for account in pd.unique(
    tx[["Source_Account","Target_Account"]].values.ravel()
):
    G.add_node(account)

for row in tx.itertuples(index=False):
    G.add_edge(
        row.Source_Account,
        row.Target_Account
    )

print("Nodes:",G.number_of_nodes())
print("Edges:",G.number_of_edges())

## 9. Graph Terminology

### Degree

Number of connections.

### In-Degree

Number of incoming relationships.

### Out-Degree

Number of outgoing relationships.

### Centrality

Measures structural importance.

### Community

A group of nodes with stronger internal connectivity.

### Path

A sequence of connected nodes.

### Component

A connected portion of the graph.

## 10. Network-Level Statistics

Graph-level statistics — node count, edge count, degree distribution and component structure — describe the shape of the transaction network and reveal whether fraud-relevant structure (dense pockets, hubs) exists at all.


In [ ]:
degrees=dict(G.degree())
in_degree=dict(G.in_degree())
out_degree=dict(G.out_degree())

print("Nodes:",G.number_of_nodes())
print("Edges:",G.number_of_edges())
print("Density:",round(nx.density(G),6))
print("Average degree:",round(np.mean(list(degrees.values())),2))

## 11. Account-Level Network Features

Per-account features such as degree, in/out flow and centrality are extracted. These network features are what let a tabular classifier see suspicious connectivity patterns.


In [ ]:
nodes=pd.DataFrame({
    "Account_ID":list(G.nodes()),
    "Degree":[degrees[n] for n in G.nodes()],
    "In_Degree":[in_degree[n] for n in G.nodes()],
    "Out_Degree":[out_degree[n] for n in G.nodes()]
})

# Transaction-based features
out_features=tx.groupby("Source_Account").agg(
    Outgoing_Tx=("Transaction_ID","count"),
    Total_Outgoing=("Amount","sum"),
    Avg_Outgoing=("Amount","mean"),
    Unique_Beneficiaries=("Target_Account","nunique"),
    Fraudulent_Outgoing=("Fraud_Label","sum")
).reset_index().rename(columns={"Source_Account":"Account_ID"})

in_features=tx.groupby("Target_Account").agg(
    Incoming_Tx=("Transaction_ID","count"),
    Total_Incoming=("Amount","sum"),
    Unique_Senders=("Source_Account","nunique")
).reset_index().rename(columns={"Target_Account":"Account_ID"})

nodes=nodes.merge(out_features,on="Account_ID",how="left")
nodes=nodes.merge(in_features,on="Account_ID",how="left")
nodes=nodes.fillna(0)

nodes["Fraud_Rate"] = (
    nodes["Fraudulent_Outgoing"] /
    nodes["Outgoing_Tx"].replace(0,np.nan)
).fillna(0)

display(nodes.head())

## 12. Centrality Features

Network position can be useful for identifying unusual or important accounts.

We calculate:

- Degree centrality
- PageRank

In a production system, other metrics may include:

- betweenness centrality,
- eigenvector centrality,
- k-core,
- motif counts,
- temporal neighborhood features.

In [ ]:
degree_centrality=nx.degree_centrality(G)
pagerank=nx.pagerank(G)

nodes["Degree_Centrality"]=nodes["Account_ID"].map(degree_centrality)
nodes["PageRank"]=nodes["Account_ID"].map(pagerank)

display(
    nodes.sort_values(
        "PageRank",
        ascending=False
    ).head(10)
)

## 13. Explore High-Connectivity Accounts

The most connected accounts are inspected directly. In fraud networks, hubs with many in-and-out flows are classic mule-account signatures worth reviewing.


In [ ]:
display(
    nodes[
        [
            "Account_ID",
            "Degree",
            "In_Degree",
            "Out_Degree",
            "Unique_Beneficiaries",
            "Unique_Senders"
        ]
    ]
    .sort_values("Degree",ascending=False)
    .head(20)
)

## 14. Network Visualization

Large banking graphs can contain millions of nodes and edges.

Therefore visualization is usually performed on:

- suspicious subgraphs,
- ego networks,
- communities,
- top-ranked accounts.

We will visualize only a small high-degree subgraph for demonstration.

In [ ]:
top_nodes=set(
    nodes.nlargest(30,"Degree")["Account_ID"]
)

subgraph=G.subgraph(top_nodes)

plt.figure(figsize=(12,9))
pos=nx.spring_layout(subgraph,seed=42)

nx.draw_networkx(
    subgraph,
    pos,
    with_labels=False,
    node_size=80,
    arrows=True
)

plt.title("Example High-Connectivity Account Network")
plt.axis("off")
plt.show()

## 15. Fraud Label at Transaction Level

The synthetic data contains a transaction-level label:

```text
Fraud_Label = 1
```

This lets us create a supervised demonstration.

In real banking environments, labels are usually difficult because:

- confirmed fraud arrives with delay,
- investigations may change labels,
- many suspicious cases are never confirmed,
- fraud patterns evolve.

In [ ]:
display(
    tx.groupby("Fraud_Label")["Amount"]
      .agg(["count","mean","median","max"])
      .round(2)
)

## 16. Aggregate Fraud Signal to Accounts

For demonstration, calculate:

```text
Fraudulent outgoing transactions
/
Total outgoing transactions
```

This is a useful analytical feature but would not be available before an investigation outcome in a real-time prediction scenario.

Therefore:

> **Do not use this feature for a production pre-transaction fraud model.**

It is included here only to illustrate network labeling and analysis.

## 17. Build Leakage-Safe Account Features

For a real prediction scenario, use only information available before the prediction timestamp.

Here we use structural and transaction-volume features:

- Degree
- In-degree
- Out-degree
- Total outgoing
- Average outgoing
- Unique beneficiaries
- Incoming transactions
- Total incoming
- Unique senders
- PageRank

In [ ]:
model_features=[
    "Degree",
    "In_Degree",
    "Out_Degree",
    "Outgoing_Tx",
    "Total_Outgoing",
    "Avg_Outgoing",
    "Unique_Beneficiaries",
    "Incoming_Tx",
    "Total_Incoming",
    "Unique_Senders",
    "Degree_Centrality",
    "PageRank"
]

# Synthetic account label: whether the account was involved
# in at least one confirmed fraudulent outgoing transaction.
nodes["Fraud_Account"]=(nodes["Fraudulent_Outgoing"]>0).astype(int)

X=nodes[model_features]
y=nodes["Fraud_Account"]

X_train,X_test,y_train,y_test=train_test_split(
    X,y,
    test_size=.20,
    stratify=y,
    random_state=42
)

print("Train:",X_train.shape)
print("Test:",X_test.shape)
print("Fraud account rate:",round(y.mean(),3))

## 18. Baseline Graph-Feature ML Model

This is an important distinction:

```text
Graph Analytics
        ↓
Create network features
        ↓
Traditional ML
        ↓
Fraud Risk
```

This is **graph-enhanced machine learning**, but not yet a Graph Neural Network.

It is often a practical starting point because network features are easier to explain and operationalize.

In [ ]:
rf=RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train,y_train)

prob=rf.predict_proba(X_test)[:,1]
pred=(prob>=.50).astype(int)

print("ROC-AUC:",round(roc_auc_score(y_test,prob),3))
print("Average Precision:",round(
    average_precision_score(y_test,prob),3
))
print()
print(classification_report(y_test,pred,digits=3))

## 19. Feature Importance

Feature importances from the classifier show which network measures contribute most to fraud detection. This tells investigators which structural signals to prioritise in future monitoring rules.


In [ ]:
importance=pd.DataFrame({
    "Feature":model_features,
    "Importance":rf.feature_importances_
}).sort_values("Importance",ascending=False)

display(importance)

plt.figure(figsize=(9,6))
sns.barplot(
    data=importance,
    x="Importance",
    y="Feature"
)
plt.title("Random Forest Feature Importance")
plt.show()

## 20. Suspicious Account Ranking

The model produces a risk score.

We can rank accounts:

```text
Account
   ↓
Fraud Risk Score
   ↓
Rank
   ↓
Investigation Queue
```

This is especially useful when investigators have limited capacity.

In [ ]:
scored=nodes[[
    "Account_ID",
    "Degree",
    "Out_Degree",
    "Incoming_Tx",
    "Outgoing_Tx"
]].copy()

scored["Fraud_Risk_Score"]=rf.predict_proba(
    nodes[model_features]
)[:,1]

scored=scored.sort_values(
    "Fraud_Risk_Score",
    ascending=False
).reset_index(drop=True)

scored["Risk_Rank"]=np.arange(1,len(scored)+1)

display(scored.head(20))

## 21. Community Detection

Fraud rings may form communities.

A community is a group of accounts with relatively dense internal relationships.

We can use community detection to identify network structures that deserve investigation.

In [ ]:
UG=G.to_undirected()

communities=nx.community.greedy_modularity_communities(UG)

community_map={}

for i,comm in enumerate(communities):
    for node in comm:
        community_map[node]=i

nodes["Community"]=nodes["Account_ID"].map(community_map)

print("Communities:",len(communities))

display(
    nodes["Community"]
    .value_counts()
    .head(10)
    .to_frame("Accounts")
)

## 22. Investigate High-Risk Communities

Instead of investigating accounts independently:

```text
Account A
Account B
Account C
```

investigators can inspect:

```text
Community / Network
       ↓
Accounts
       ↓
Transactions
       ↓
Common counterparties
       ↓
Potential ring
```

This can make network investigation more efficient.

In [ ]:
community_risk=(
    nodes.groupby("Community")
    .agg(
        Accounts=("Account_ID","count"),
        Avg_Risk=("Fraud_Risk_Score","mean"),
        Max_Risk=("Fraud_Risk_Score","max"),
        Fraud_Accounts=("Fraud_Account","sum")
    )
    .sort_values("Avg_Risk",ascending=False)
)

display(community_risk.head(15).round(3))

## 23. Fraud Ring Subgraph

Select a high-risk community and visualize its network.

This is not proof of fraud.

It is an **investigation aid**.

In [ ]:
top_community=community_risk.index[0]

community_nodes=set(
    nodes.loc[
        nodes["Community"]==top_community,
        "Account_ID"
    ]
)

community_graph=G.subgraph(community_nodes)

plt.figure(figsize=(12,9))

pos=nx.spring_layout(
    community_graph,
    seed=42
)

node_risk=[
    scored.set_index("Account_ID")
          .loc[n,"Fraud_Risk_Score"]
    for n in community_graph.nodes()
]

nx.draw_networkx(
    community_graph,
    pos,
    with_labels=False,
    node_size=120,
    node_color=node_risk,
    cmap="Reds",
    arrows=True
)

plt.title("High-Risk Community Network")
plt.axis("off")
plt.show()

## 24. What Makes This "Graph ML"?

There are several levels.

### Level 1 — Graph Analytics

```text
Network
 ↓
Degree
Centrality
Community
```

### Level 2 — Graph Features + ML

```text
Graph
 ↓
Network Features
 ↓
Random Forest / XGBoost / Logistic Regression
```

### Level 3 — Graph Neural Networks

```text
Graph
 ↓
Node / Edge Features
 ↓
Message Passing
 ↓
Graph Neural Network
 ↓
Fraud Prediction
```

This notebook demonstrates Levels 1 and 2 and explains the path toward Level 3.

## 25. Graph Neural Network Concept

A Graph Neural Network allows a node to learn from its neighborhood.

Conceptually:

```text
        B
       ↙ ↘
      A   C
       ↘ ↙
        D
```

For account A:

```text
A's own features
+
B's features
+
C's features
+
relationship structure
        ↓
GNN representation
        ↓
Fraud prediction
```

This is called **message passing**.

## 26. Possible GNN Models

For production research, common approaches include:

### GCN

Graph Convolutional Network

### GraphSAGE

Samples and aggregates neighborhood information.

### GAT

Graph Attention Network.

### GNN for Link Prediction

Predict suspicious relationships:

```text
Should A → B be considered suspicious?
```

### GNN for Node Classification

Predict:

```text
Is account A suspicious?
```

### GNN for Graph Classification

Predict whether an entire subgraph represents a fraud pattern.

## 27. Fraud Detection as Node Classification

A common Graph ML formulation:

```text
Node
=
Bank Account

Features
=
Account behavior

Edges
=
Transactions

Target
=
Fraud / Normal
```

The GNN learns:

```text
Account behavior
+
Neighborhood behavior
+
Network structure
        ↓
Fraud probability
```

This is often more expressive than treating every account independently.

## 28. Fraud Detection as Link Prediction

Another formulation:

```text
A → B
```

The model predicts whether the relationship is suspicious.

Useful when the primary question is:

> **"Is this transaction / relationship suspicious?"**

Features can include:

- transaction amount,
- frequency,
- historical relationship,
- shared counterparties,
- neighborhood embeddings,
- temporal behavior.

## 29. Temporal Graph Challenge

Banking networks change over time.

Example:

```text
January
A → B

February
A → C

March
A → D

April
B → D
```

A production system should preserve temporal information.

Otherwise:

> Future transactions can accidentally leak into historical predictions.

This is a major issue in graph fraud detection.

## 30. Evaluation

Useful metrics include:

### Classification

- ROC-AUC
- PR-AUC
- Precision
- Recall
- F1

### Ranking

- Precision@K
- Recall@K
- Lift@K

### Graph-specific operational metrics

- suspicious ring detection rate
- investigation yield
- confirmed fraud amount
- false-positive investigation volume
- alert reduction
- time-to-detection

## 31. Why PR-AUC Matters

Fraud is often rare.

Imagine:

```text
100,000 transactions
99,000 normal
1,000 fraud
```

Accuracy can be misleading.

A model predicting everything as normal would have:

```text
99% accuracy
```

but:

```text
0 fraud detected
```

Therefore precision-recall metrics are often important for imbalanced fraud problems.

## 32. Production Architecture

```text
Transaction Stream
        ↓
Graph Construction
        ↓
Graph Feature Store
        ↓
Graph Analytics
        ↓
Graph ML / GNN
        ↓
Fraud Risk Score
        ↓
Rules + Risk Engine
        ↓
Alert / Block / Review
        ↓
Fraud Investigation
        ↓
Confirmed Labels
        ↓
Model Retraining
```

For real-time fraud, the graph must support continuously changing relationships.

## 33. Important Banking Controls

Fraud detection is a high-impact banking system.

Production implementation requires:

- explainability,
- audit trails,
- data governance,
- access control,
- privacy protection,
- model monitoring,
- human investigation,
- false-positive management,
- regulatory compliance.

A model score should generally be treated as an input to an investigation/risk process rather than automatically treated as proof of fraud.

## 34. Common Mistakes

1. Treating a graph as a simple table.
2. Ignoring edge direction.
3. Ignoring time.
4. Data leakage through future transactions.
5. Using confirmed fraud labels incorrectly.
6. Over-relying on centrality.
7. Assuming a highly connected account is fraudulent.
8. Ignoring false positives.
9. Evaluating only ROC-AUC.
10. Treating network association as proof of fraud.

## 35. Final Executive Summary

### Business Question

> **Can relationships between accounts help identify fraud?**

### Graph Representation

```text
Accounts
   ↓
Nodes

Transactions
   ↓
Edges
```

### Analytical Pipeline

```text
Transactions
     ↓
Network
     ↓
Graph Features
     ↓
Community Detection
     ↓
ML Risk Model
     ↓
Fraud Risk Ranking
     ↓
Suspicious Network Investigation
```

### Advanced Direction

```text
Graph Features + ML
        ↓
Graph Neural Network
        ↓
Node / Edge / Subgraph Fraud Detection
```

### Main Takeaway

> **Graph ML adds relationship intelligence to fraud detection.**

Traditional ML asks:

> "Does this account/transaction look suspicious?"

Graph ML adds:

> **"Does this account/transaction look suspicious given its relationships and surrounding network?"**

That network perspective is particularly valuable for detecting coordinated activity and potential fraud rings.